# ⚡ Notebook 02: Thực nghiệm Apriori vs FP-Growth (mlxtend) & Trích xuất Luật

Mục tiêu:
1. Nạp tập giao dịch từ file `.parquet`.
2. Chạy Apriori và FP-Growth của `mlxtend` trên cùng đầu vào.
3. Đối chiếu: hai thuật toán phải cho cùng tập phổ biến.
4. Đọc và vẽ kết quả benchmark (`results/benchmark_results.csv`).
5. Sinh luật kết hợp (Confidence ≥ 20%, Lift ≥ 1.5).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from mining import load_transactions, encode_transactions, mine_frequent_itemsets, mine_rules

In [ ]:
# Nạp dữ liệu mẫu 10k và đối chiếu hai thuật toán
parquet_path = Path('../data/processed/transactions_product_10k.parquet')
MIN_SUPPORT = 0.01

transactions = load_transactions(parquet_path)
onehot = encode_transactions(transactions, MIN_SUPPORT)
print(f"{len(transactions):,} giao dịch, {onehot.shape[1]} món sau khi cắt tỉa")

fi_apriori = mine_frequent_itemsets('apriori', onehot, MIN_SUPPORT)
fi_fpgrowth = mine_frequent_itemsets('fpgrowth', onehot, MIN_SUPPORT)

as_set = lambda df: {(frozenset(s), round(v, 10)) for s, v in zip(df['itemsets'], df['support'])}
print('Cùng kết quả:', as_set(fi_apriori) == as_set(fi_fpgrowth))

In [ ]:
# Luật kết hợp
rules = mine_rules(fi_fpgrowth, min_confidence=0.2, min_lift=1.5)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)